In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim import lr_scheduler

import numpy as np
import torchvision
from torchvision import datasets, models, transforms
import matplotlib.pyplot as plt
import time
import os
import copy

In [3]:
import os
import shutil
from random import shuffle

# Paths to the dataset directory and the train/val directories
dataset_path = '/Users/Elizabeth/Desktop/ML/animals copy'
train_path = 'train'
val_path = 'val'

# Create train and val directories if they don't exist
os.makedirs(train_path, exist_ok=True)
os.makedirs(val_path, exist_ok=True)

# List all class directories in the dataset
class_directories = [d for d in os.listdir(dataset_path) if os.path.isdir(os.path.join(dataset_path, d))]

# Iterate over each class directory
for class_dir in class_directories:
    class_path = os.path.join(dataset_path, class_dir)
    images = os.listdir(class_path)
    shuffle(images)  # Shuffle to randomize the files that go into train and val

    # Ensure the destination directories for the class exist
    train_class_path = os.path.join(train_path, class_dir)
    val_class_path = os.path.join(val_path, class_dir)
    os.makedirs(train_class_path, exist_ok=True)
    os.makedirs(val_class_path, exist_ok=True)

    # Move the first 40 images to the train directory and the rest to the val directory
    for i, image in enumerate(images):
        if i < 40:
            shutil.move(os.path.join(class_path, image), os.path.join(train_class_path, image))
        else:
            shutil.move(os.path.join(class_path, image), os.path.join(val_class_path, image))

In [4]:
data_dir = '/Users/Elizabeth/Desktop/ML/animals_split'
train_dir = '/Users/Elizabeth/Desktop/ML/train'
val_dir = '/Users/Elizabeth/Desktop/ML/val'

In [5]:
data_transforms = {
    'train': transforms.Compose([
        transforms.RandomResizedCrop(224),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'val': transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}


In [6]:
image_datasets = {x: datasets.ImageFolder(os.path.join(data_dir, x),
                                          data_transforms[x])
                  for x in ['train', 'val']}


In [7]:
dataloaders = {x: torch.utils.data.DataLoader(image_datasets[x], batch_size=4,
                                             shuffle=True, num_workers=4)
              for x in ['train', 'val']}


In [8]:
dataset_sizes = {x: len(image_datasets[x]) for x in ['train', 'val']}


In [9]:
class_names = image_datasets['train'].classes

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [10]:
def train_model(model, criterion, optimizer, scheduler, num_epochs=25):
    t1 = time.time()

    best_model_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0

    for epoch in range(num_epochs):
        print(f'Epoch {epoch}/{num_epochs - 1}')
        print('-' * 10)

        # Each epoch has a training and validation phase
        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()  # Set model to training mode
            else:
                model.eval()   # Set model to evaluate mode

            running_loss = 0.0
            running_corrects = 0

            # Iterate over data.
            for inputs, labels in dataloaders[phase]:
                inputs = inputs.to(device)
                labels = labels.to(device)

                # zero the parameter gradients
                optimizer.zero_grad()

                # Forward
                # Track history if only in train
                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)

                    # backward + optimize only if in training phase
                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                # statistics
                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

            if phase == 'train':
                scheduler.step()

            epoch_loss = running_loss / dataset_sizes[phase]
            epoch_acc = running_corrects / dataset_sizes[phase]

            print(f'{phase} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

            # deep copy the model
            if phase == 'val' and epoch_acc > best_acc:
                best_acc = epoch_acc
                best_model_wts = copy.deepcopy(model.state_dict())

        print()

    time_elapsed = time.time() - t1
    print(f'Training complete in {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s')
    print(f'Best val Acc: {best_acc:4f}')

    # load best model weights
    model.load_state_dict(best_model_wts)
    return model

In [11]:
def visualize_model(model, num_images=6):
    was_training = model.training
    model.eval()
    images_so_far = 0
    fig = plt.figure()

    with torch.no_grad():
        for i, (inputs, labels) in enumerate(dataloaders['val']):
            inputs = inputs.to(device)
            labels = labels.to(device)

            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)

            for j in range(inputs.size()[0]):
                images_so_far += 1
                ax = plt.subplot(num_images//2, 2, images_so_far)
                ax.axis('off')
                ax.set_title(f'predicted: {class_names[preds[j]]}')
                imshow(inputs.cpu().data[j])

                if images_so_far == num_images:
                    model.train(mode=was_training)
                    return
        model.train(mode=was_training)

In [12]:
input_shape = 3
hidden_units = 64
block_1 = nn.Sequential(nn.Conv2d(in_channels=input_shape,
                                               out_channels=hidden_units,
                                               kernel_size=5, # how big is the square that's going over the image?
                                               stride=1, # default
                                               padding=1), # Options = "valid" (no padding) or "same" (the output has the same shape as the input) or int for specific number
                                     nn.BatchNorm2d(hidden_units,
                                                    eps=1e-05,
                                                    momentum=0.1,
                                                    affine=True,
                                                    track_running_stats=True),
                                     nn.ReLU(),
                                     nn.Conv2d(in_channels=hidden_units,
                                               out_channels=hidden_units,
                                               kernel_size=5,
                                               stride=1,
                                               padding=1),
                                     nn.BatchNorm2d(hidden_units,
                                                    eps=1e-05,
                                                    momentum=0.1,
                                                    affine=True,
                                                    track_running_stats=True),
                                     nn.ReLU(),
                                     nn.MaxPool2d(kernel_size=2, stride=2)
                                     )

block_2 = nn.Sequential(nn.Conv2d(in_channels=hidden_units,
                                               out_channels=hidden_units,
                                               kernel_size=5,
                                               padding=1),
                                     nn.BatchNorm2d(hidden_units,
                                                    eps=1e-05,
                                                    momentum=0.1,
                                                    affine=True,
                                                    track_running_stats=True),
                                     nn.ReLU(),
                                     nn.Conv2d(in_channels=hidden_units,
                                               out_channels=hidden_units,
                                               kernel_size=5,
                                               padding=1),
                                     nn.BatchNorm2d(hidden_units,
                                                    eps=1e-05,
                                                    momentum=0.1,
                                                    affine=True,
                                                    track_running_stats=True),
                                     nn.ReLU(),
                                     nn.MaxPool2d(kernel_size=2, stride=2)
        )


image = torch.randn(size=(1, 3, 224, 224)) # [batch_size, color_channels, height, width]

x = block_1(image)

x = block_2(x)

x.shape

torch.Size([1, 64, 53, 53])

In [13]:
# Create a simple CNN
class SimpleCNN(nn.Module):

    def __init__(self, input_shape: int, hidden_units: int, output_shape: int):
        super().__init__()
        self.block_1 = nn.Sequential(nn.Conv2d(in_channels=input_shape,
                                               out_channels=hidden_units,
                                               kernel_size=5, # how big is the square that's going over the image?
                                               stride=1, # default
                                               padding=1), # Options = "valid" (no padding) or "same" (the output has the same shape as the input) or int for specific number
                                     nn.BatchNorm2d(hidden_units,
                                                    eps=1e-05,
                                                    momentum=0.1,
                                                    affine=True,
                                                    track_running_stats=True),
                                     nn.ReLU(),
                                     nn.Conv2d(in_channels=hidden_units,
                                               out_channels=hidden_units,
                                               kernel_size=5,
                                               stride=1,
                                               padding=1),
                                     nn.BatchNorm2d(hidden_units,
                                                    eps=1e-05,
                                                    momentum=0.1,
                                                    affine=True,
                                                    track_running_stats=True),
                                     nn.ReLU(),
                                     nn.MaxPool2d(kernel_size=2, stride=2)
                                     )

        self.block_2 = nn.Sequential(nn.Conv2d(in_channels=hidden_units,
                                               out_channels=hidden_units,
                                               kernel_size=5,
                                               padding=1),
                                     nn.BatchNorm2d(hidden_units,
                                                    eps=1e-05,
                                                    momentum=0.1,
                                                    affine=True,
                                                    track_running_stats=True),
                                     nn.ReLU(),
                                     nn.Conv2d(in_channels=hidden_units,
                                               out_channels=hidden_units,
                                               kernel_size=5,
                                               padding=1),
                                     nn.BatchNorm2d(hidden_units,
                                                    eps=1e-05,
                                                    momentum=0.1,
                                                    affine=True,
                                                    track_running_stats=True),
                                     nn.ReLU(),
                                     nn.MaxPool2d(kernel_size=2, stride=2)
        )

        self.classifier = nn.Sequential(nn.Flatten(), # Each layer of our network compresses and changes the shape of our input data.
                                        nn.Linear(in_features=hidden_units*53*53,
                                                  out_features=512),
                                        nn.ReLU(),
                                        nn.Linear(in_features=512,
                                                  out_features=output_shape),
                                        )

    def forward(self, x: torch.Tensor):
        x = self.block_1(x)
        # print(x.shape)
        x = self.block_2(x)
        # print(x.shape)
        x = self.classifier(x)
        # print(x.shape)
        return x

cnn = SimpleCNN(input_shape=3, hidden_units=10, output_shape=2).to(device)
cnn

SimpleCNN(
  (block_1): Sequential(
    (0): Conv2d(3, 10, kernel_size=(5, 5), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(10, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Conv2d(10, 10, kernel_size=(5, 5), stride=(1, 1), padding=(1, 1))
    (4): BatchNorm2d(10, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): ReLU()
    (6): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (block_2): Sequential(
    (0): Conv2d(10, 10, kernel_size=(5, 5), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(10, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Conv2d(10, 10, kernel_size=(5, 5), stride=(1, 1), padding=(1, 1))
    (4): BatchNorm2d(10, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): ReLU()
    (6): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten

In [15]:
criterion = nn.CrossEntropyLoss()

# Observe that only parameters of final layer are being optimized as
# opposed to before.
optimizer = optim.Adam(cnn.parameters(), lr=0.001)

# Decay LR by a factor of 0.1 every 15 epochs
exp_lr_scheduler = lr_scheduler.StepLR(optimizer, step_size=15, gamma=0.1)

cnn = train_model(cnn, criterion, optimizer, exp_lr_scheduler, num_epochs=15)

Epoch 0/14
----------


IndexError: Target 61 is out of bounds.